In [21]:
from flask import Flask, request
import folium
import webbrowser
import os
import threading
import time
import requests
import speech_recognition as sr
import psutil
import ctypes



def listen_command():
    r = sr.Recognizer()

    with sr.Microphone() as source:
        print("\n Say: 'get my location'")
        r.adjust_for_ambient_noise(source)
        audio = r.listen(source)

    try:
        command = r.recognize_google(audio).lower()
        print("You said:", command)
        return command
    except:
        print("Could not understand voice")
        return ""

app = Flask(__name__)

# This will store GPS from browser
coords = {"lat": None, "lon": None}


@app.route("/")
def home():
    return """
    <h2>Getting your location...</h2>

    <script>
        navigator.geolocation.getCurrentPosition(function(position) {
            fetch(`/save?lat=${position.coords.latitude}&lon=${position.coords.longitude}`)
            .then(() => {
                document.body.innerHTML = "Location received! You can close this tab.";
            });
        });
    </script>
    """


@app.route("/save")
def save():
    global coords
    coords["lat"] = float(request.args.get("lat"))
    coords["lon"] = float(request.args.get("lon"))
    return "OK"


def get_clean_address(lat, lon):
    url = "https://nominatim.openstreetmap.org/reverse"
    params = {
        "lat": lat,
        "lon": lon,
        "format": "json",
        "addressdetails": 1
    }
    headers = {
        "User-Agent": "gps-app"
    }

    response = requests.get(url, params=params, headers=headers)
    data = response.json()

    address = data.get("address", {})

    building = address.get("building")
    house = address.get("house_number")
    road = address.get("road")

    parts = []

    if building:
        parts.append(building)

    if house and road:
        parts.append(f"{house} {road}")
    elif road:
        parts.append(road)

    return ", ".join(parts) if parts else data.get("display_name", "Unknown")


def create_map(lat, lon):
    address = get_clean_address(lat, lon)

    m = folium.Map(location=[lat, lon], zoom_start=15)
    folium.Marker([lat, lon], popup=address).add_to(m)

    file_path = os.path.abspath("real_location_map.html")
    m.save(file_path)

    webbrowser.open("file:///" + file_path)


def run_server():
    app.run(port=5000)


def check_battery():
    battery = psutil.sensors_battery()

    if battery is None:
        print("Battery info not available")
        return

    percent = battery.percent
    plugged = battery.power_plugged

    print(f"🔋 Battery: {percent}%")

    if percent <= 20 and not plugged:
        print("⚠️ WARNING: Battery is below 20%!")
        
    if percent <= 20 and not plugged:
        ctypes.windll.user32.MessageBoxW(
            0,
            "Battery is below 20%! Please plug in your charger.",
            "Battery Warning",
            1
        )





if __name__ == "__main__":

    check_battery()

    command = listen_command()

    if "location" in command:

        threading.Thread(target=run_server).start()

        webbrowser.open("http://127.0.0.1:5000/")

        print("Waiting for GPS...")

        while coords["lat"] is None:
            time.sleep(1)

        print("Got location!")

        address = get_clean_address(coords["lat"], coords["lon"])

        print("\n Clean Location:")
        print(address)

        create_map(coords["lat"], coords["lon"])

    else:
        print("Voice command not recognized. Say 'get my location'")



🔋 Battery: 99%

 Say: 'get my location'
Could not understand voice
Voice command not recognized. Say 'get my location'
